In [ ]:
# Import all basic functions
import onnx
import onnxruntime as ort
import ml_dtypes
import numpy as np
import h5py
import torch

# Import config
from config import load_config, InputSourceConfig

import importlib
import sys
import tqdm

In [ ]:
# data generation
from generate_data.datagen import (
    sample_ambient_points,
    sample_around_points,
    repeat_points_to_match,
    get_token_ids_array,
)

# model generation
from construct.construct import (
    ensure_base_model,
    ensure_subgraph_pullback,
    ensure_subgraphs,
)

# execution code
from execute.optimization_loop import optimization_loop, NewtonRaphson

# 1. Reload the parent module
importlib.reload(sys.modules["execute.optimization_loop"])
importlib.reload(sys.modules["generate_data.datagen"])
importlib.reload(sys.modules["construct.construct"])


In [ ]:
cfg = load_config("configs/qwen_jac_gather.toml")
force = False


# makes sure that our base model is loaded
ensure_base_model(cfg, force=force, verbose=False)
sub_graphs = ensure_subgraphs(cfg, force=force)

# Define output session
sess_options = ort.SessionOptions()

# Very Important --- will cause errors if missing
sess_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_BASIC
sess_options.intra_op_num_threads = 8  # pick a value <= your usable core count

print(sub_graphs[0][0])
embedding_gen = ort.InferenceSession(
    str(sub_graphs[0][0]),
    sess_options=sess_options,
    providers=["CUDAExecutionProvider"],
)
output_names = {o.name for o in embedding_gen.get_outputs()}
input_names = {o.name for o in embedding_gen.get_inputs()}
input_names = [i for i in input_names]
print(input_names, output_names)


In [ ]:
# Load config
cfg = load_config("configs/qwen_jac_sublayer_tokens_new.toml")

# Re-use all existing models
force = False

# makes sure that our base model is loaded
ensure_base_model(cfg, force=force, verbose=False)

force = True
# makes sure that out subgraphs are present
sub_graphs = ensure_subgraphs(cfg, force=force)

# makes sure that we have the jacobains
sub_graphs_pullbacks = ensure_subgraph_pullback(
    sub_graphs,
    force=force,
)


In [ ]:
out_sess = ort.InferenceSession(
    str(sub_graphs_pullbacks[0]),
    sess_options=sess_options,
    providers=["CUDAExecutionProvider"],
)


# Get all output and input names
output_names = {o.name for o in out_sess.get_outputs()}
input_names = {o.name for o in out_sess.get_inputs()}
input_names = [i for i in input_names]


print(output_names, input_names)


In [ ]:
inputs = get_token_ids_array(
    cfg.input_source,
    cfg.seq,
)
new_input = np.repeat(
    inputs.reshape(1, *inputs.shape),
    cfg.vocab_size,
    axis=0,
)
new_input[:, :, -1] = np.arange(cfg.vocab_size).reshape(-1, 1)

In [ ]:
x_out = []
for s in tqdm.tqdm(range(new_input.shape[0])):
    inputs = new_input[s]
    outputs = embedding_gen.run(None, {"input_ids": inputs})[0].astype(np.float32)
    x_out.append(outputs)
    


In [ ]:
%load_ext line_profiler

In [ ]:
y_out = []
for s in tqdm.tqdm(range(3)):
    
    y0 = sample_around_points(x_out[s], n_samples=1000, max_radius=1e-3).astype(ml_dtypes.bfloat16)
    # print(y0.shape)
        
      


In [ ]:
import matplotlib.pyplot as plt

x32 = x.astype(np.float32)
fx = out_sess.run(None, {input_names[0]: x})[0].astype(np.float32)

dx_list = []
dfx_list = []
for s in range(y0.shape[0]):
    y_s = y0[s]
    fy_s = out_sess.run(None, {input_names[0]: y_s})[0].astype(np.float32)
    dx_list.append(np.linalg.norm(y_s.astype(np.float32) - x32, axis=-1))
    dfx_list.append(np.linalg.norm(fy_s - fx, axis=-1))

dx = np.concatenate([d.ravel() for d in dx_list])
dfx = np.concatenate([d.ravel() for d in dfx_list])


In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(dx, dfx, s=4, alpha=0.3)
plt.xlabel("||y - x||")
plt.ylabel("||f(y) - f(x)||")
plt.title("||f(y) - f(x)|| vs ||y - x||")
plt.xlim(0, 0.001)
plt.ylim(0, 0.001)
plt.tight_layout()
plt.savefig("lipschitz_scatter.png", dpi=150)

In [ ]:
plt.plot(np.sort(dfx))
len(dfx[dfx == 0.0])

In [ ]:
plt.plot(dx[dfx == 0.0])

In [ ]:
What is the relative positioning of all of these points?
# From the zero ids go back to vectors and then from there figure out how is this related to the kernel of the jacobian

    